In [1]:
import pandas as pd
from datasets import Dataset

In [3]:
df = pd.read_csv("review.csv")
dataset = Dataset.from_pandas(df)

In [4]:
from datasets import load_dataset

In [5]:
dataset = load_dataset("csv", data_files="review.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
dataset_json = load_dataset("json", data_files="data.json")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
dataset_json

Dataset({
    features: ['text', 'label'],
    num_rows: 2
})

In [8]:
data = {
    "text": [
        "这家餐厅味道很好，推荐！",
        "服务态度差，菜品也不新鲜。",
        "环境优雅，适合约会。",
        "价格太贵，性价比低。",
        "分量足，味道正宗。",
        "等了一个小时才上菜，体验很差。",
    ],
    "label": [1, 0, 1, 0, 1, 0],
}

In [10]:
data

{'text': ['这家餐厅味道很好，推荐！',
  '服务态度差，菜品也不新鲜。',
  '环境优雅，适合约会。',
  '价格太贵，性价比低。',
  '分量足，味道正宗。',
  '等了一个小时才上菜，体验很差。'],
 'label': [1, 0, 1, 0, 1, 0]}

In [11]:
dataset_pythondict = Dataset.from_dict(data)

In [12]:
dataset_pythondict

Dataset({
    features: ['text', 'label'],
    num_rows: 6
})

In [13]:
type(dataset_pythondict)

datasets.arrow_dataset.Dataset

In [14]:
split = dataset_pythondict.train_test_split(test_size=0.2, seed=42)

In [15]:
train_dataset = split["train"]
test_dataset = split["test"]

In [16]:
print(f"训练集: {len(train_dataset)} 条")
print(f"测试集: {len(test_dataset)} 条")

训练集: 4 条
测试集: 2 条


In [17]:
from transformers import AutoTokenizer

In [18]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

In [19]:
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [20]:
train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [21]:
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [22]:
train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4
})

In [24]:
import torch

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [25]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese", num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
training_args = TrainingArguments(
    output_dir="./output832",
    num_train_epochs=30,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=5,
)

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.603611
2,No log,0.588929
3,No log,0.583518
4,No log,0.554356
5,0.444900,0.519514
6,0.444900,0.422798
7,0.444900,0.381989
8,0.444900,0.420246
9,0.444900,0.411832
10,0.195700,0.401379


TrainOutput(global_step=30, training_loss=0.16234432458877562, metrics={'train_runtime': 3.3034, 'train_samples_per_second': 36.326, 'train_steps_per_second': 9.082, 'total_flos': 7893331660800.0, 'train_loss': 0.16234432458877562, 'epoch': 30.0})

In [29]:
trainer.save_model("./my-custom-model")